In [10]:
%pip install tensorflow
%pip install keras
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import numpy as np
import re
import pickle

from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout # type: ignore
from tensorflow.keras.preprocessing.text import Tokenizer # type: ignore
import tensorflow.keras.preprocessing.sequence # type: ignore
from tensorflow.keras.preprocessing.sequence import pad_sequences  # type: ignore
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


In [12]:
# loading data
df = pd.read_csv('spam.csv', encoding='latin-1')

df = df[['v1', 'v2']]
df.columns = ['label', 'message']

encoder = LabelEncoder()
df['label_num'] = encoder.fit_transform(df['label'])


In [13]:
#cleaning
def cleaning(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text

df['clean_message'] = df['message'].apply(cleaning)


In [14]:
# token
size = 10000  
max_length = 150    
trunc_type = 'post'
padding_type = 'post'
oov_tok = "<OOV>"   

tokenizer = Tokenizer(num_words=size, oov_token=oov_tok)
tokenizer.fit_on_texts(df['clean_message'])

sequences = tokenizer.texts_to_sequences(df['clean_message'])
padded_sequences = pad_sequences(sequences, maxlen=max_length, padding=padding_type, truncating=trunc_type)

In [15]:
#cnn model and training
embedding_dim = 64

model = Sequential([
    Embedding(size, embedding_dim, input_length=max_length),
    
    Conv1D(128, 5, activation='relu'),
    
    GlobalMaxPooling1D(),
    
    Dense(64, activation='relu'),
    Dropout(0.5), 
    
    Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

#training
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, df['label_num'], test_size=0.2, random_state=42)

history = model.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test), batch_size=32)

#model save
import pickle
with open('cnn_model.keras', 'wb') as f:
    model.save('cnn_model.keras')
with open('cnn_tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

c:\Users\marou\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_1          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8802 - loss: 0.3296 - val_accuracy: 0.9785 - val_loss: 0.0859
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9845 - loss: 0.0596 - val_accuracy: 0.9857 - val_loss: 0.0586
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9969 - loss: 0.0156 - val_accuracy: 0.9848 - val_loss: 0.0587
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9993 - loss: 0.0055 - val_accuracy: 0.9848 - val_loss: 0.0670
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9996 - loss: 0.0027 - val_accuracy: 0.9865 - val_loss: 0.0741


In [16]:
cnn_preds = model.predict(X_test)
cnn_accuracy = np.mean((cnn_preds > 0.5).flatten() == y_test)
print(f"CNN Accuracy: {cnn_accuracy:.4f} ({cnn_accuracy*100:.2f}%)")

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
CNN Accuracy: 0.9865 (98.65%)


In [17]:
def predict_spam_cnn(input_text):
    cleaned_text = cleaning(input_text)
    
    sequence = tokenizer.texts_to_sequences([cleaned_text])
    
    padded_sequence = pad_sequences(sequence, maxlen=max_length, padding=padding_type, truncating=trunc_type)
    
    prediction_prob = model.predict(padded_sequence, verbose=0)[0][0]
    
    if prediction_prob >= 0.5:
        return "Spam"
    else:
        return "Not Spam"

test_msg = "hey my name is Tristan and you just won a free ticket to the Bahamas! Click here to claim now!"
print(f"\nTest Prediction for: '{test_msg}'")
print(f"Result: {predict_spam_cnn(test_msg)}")


Test Prediction for: 'hey my name is Tristan and you just won a free ticket to the Bahamas! Click here to claim now!'
Result: Spam


In [18]:
import os
print(os.getcwd())

c:\Cpyprojects\NLP\NLP-spam-detection
